# Hansen Ch.19 Nonparametric Regression

理论逐步证明见同目录 md（**19.1–19.11**）。本 notebook：Gaussian 核 NW/LL + ROT 带宽 + 点态 SE。

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import pinv

ROOT = Path("../..") / "hansen" / "econometrics" / "data"  # relative to docs/chXX/
def gaussian_K(u):
    return np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)

def rot_bandwidth(x, y, q=4):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    xi1, xi2 = np.quantile(x, 0.05), np.quantile(x, 0.95)
    X = np.column_stack([x**j for j in range(q + 1)])
    b = pinv(X.T @ X) @ (X.T @ y)
    e = y - X @ b
    s2 = np.mean(e**2)

    def m2(xx):
        s = 0.0
        for j in range(2, q + 1):
            s += j * (j - 1) * b[j] * xx ** (j - 2)
        return s

    w = (x >= xi1) & (x <= xi2)
    Bhat = np.mean((0.5 * m2(x[w])) ** 2)
    Bhat = max(Bhat, 1e-12)
    n = len(x)
    h = 0.58 * (s2 * (xi2 - xi1) / (n * Bhat)) ** (1 / 5)
    return float(h)

def nw_est(x, y, grid, h):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    out = []
    for g in grid:
        w = gaussian_K((x - g) / h)
        sw = w.sum()
        out.append(np.nan if sw < 1e-12 else np.sum(w * y) / sw)
    return np.array(out)

def ll_est(x, y, grid, h):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    out = []
    for g in grid:
        w = gaussian_K((x - g) / h)
        X = np.column_stack([np.ones(len(x)), x - g])
        XtWX = X.T @ (w[:, None] * X)
        XtWy = X.T @ (w * y)
        b = pinv(XtWX) @ XtWy
        out.append(b[0])
    return np.array(out)

def se_local(x, y, grid, h, mhat, which="nw"):
    RK = 1 / (2 * np.sqrt(np.pi))
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)
    m_at = nw_est(x, y, x, h) if which == "nw" else ll_est(x, y, x, h)
    e2 = (y - m_at) ** 2
    ses = []
    for g in grid:
        w = gaussian_K((x - g) / h)
        sw = w.sum()
        if sw < 1e-8:
            ses.append(np.nan)
            continue
        fhat = sw / (n * h)
        s2 = max(np.sum(w * e2) / sw, 1e-12)
        ses.append(np.sqrt(RK * s2 / (max(fhat, 1e-12) * n * h)))
    return np.array(ses)

def report(name, x, y, grid):
    h = rot_bandwidth(x, y)
    nw = nw_est(x, y, grid, h)
    ll = ll_est(x, y, grid, h)
    se_n = se_local(x, y, grid, h, nw, "nw")
    se_l = se_local(x, y, grid, h, ll, "ll")
    print(f"{name}: n={len(x)}, h_rot={h:.4f}")
    idx = np.linspace(0, len(grid) - 1, 6, dtype=int)
    print("     x      NW    se_NW      LL    se_LL")
    for i in idx:
        print(f"{grid[i]:7.3f} {nw[i]:7.3f} {se_n[i]:7.3f} {ll[i]:7.3f} {se_l[i]:7.3f}")
    return h, grid, nw, ll, se_n, se_l


## 19.7 DDK boys + tracking

In [ ]:
ddk = pd.read_excel(ROOT / "DDK2011/DDK2011.xlsx")
boys = ddk[(ddk["girl"] == 0) & (ddk["tracking"] == 1)].copy()
x = pd.to_numeric(boys["percentile"], errors="coerce")
y = pd.to_numeric(boys["r2_totalscore"], errors="coerce")
m = np.isfinite(x) & np.isfinite(y)
x, y = x[m].values, y[m].values
grid = np.linspace(max(1, x.min()), min(99, x.max()), 40)
report("19.7 boys", x, y, grid)


## 19.8 CPS education=20

In [ ]:
cps = pd.read_excel(ROOT / "cps09mar/cps09mar.xlsx")
cps["experience"] = cps["age"] - cps["education"] - 6
cps["lwage"] = np.log(cps["earnings"] / (cps["hours"] * cps["week"]))
sub = cps[(cps.education == 20) & (cps.experience >= 0) & (cps.experience <= 40)].copy()
sub = sub.replace([np.inf, -np.inf], np.nan).dropna(subset=["lwage", "experience"])
for sex, lab in [(0, "men"), (1, "women")]:
    s = sub[sub.female == sex]
    report(f"19.8 {lab}", s.experience.values.astype(float), s.lwage.values.astype(float), np.linspace(0, 40, 41))


## 19.9 Invest I on Q (Q<=5)

In [ ]:
inv = pd.read_excel(ROOT / "Invest1993/Invest1993.xlsx")
inv["I"] = pd.to_numeric(inv["inva"], errors="coerce")
inv["Q"] = pd.to_numeric(inv["vala"], errors="coerce")
s = inv[(inv.Q <= 5) & inv.I.notna() & inv.Q.notna()]
if len(s) > 8000:
    s = s.sample(8000, random_state=1)
report("19.9", s.Q.values, s.I.values, np.linspace(0.05, 5, 40))


## 19.10 RR2010

In [ ]:
rr = pd.read_excel(ROOT / "RR2010/RR2010.xlsx")
x = pd.to_numeric(rr["debt"], errors="coerce").values
y = pd.to_numeric(rr["gdp"], errors="coerce").values
m = np.isfinite(x) & np.isfinite(y)
grid = np.linspace(np.percentile(x[m], 2), np.percentile(x[m], 98), 40)
report("19.10 debt", x[m], y[m], grid)
xi = pd.to_numeric(rr["inflation"], errors="coerce").values
m2 = np.isfinite(xi) & np.isfinite(y)
gridi = np.linspace(np.percentile(xi[m2], 5), np.percentile(xi[m2], 95), 30)
report("19.10 inflation", xi[m2], y[m2], gridi)


## 19.11 Nonlinear AR for GDP growth

In [ ]:
qd = pd.read_excel(ROOT / "FRED-QD/FRED-QD.xlsx")
g = pd.to_numeric(qd["gdpc1"], errors="coerce")
Y = 100 * ((g / g.shift(1)) ** 4 - 1)
df = pd.DataFrame({"Y": Y, "Ylag": Y.shift(1)}).dropna()
grid = np.linspace(np.percentile(df.Ylag, 5), np.percentile(df.Ylag, 95), 40)
report("19.11", df.Ylag.values, df.Y.values, grid)
